# MotorAssistEnv — Disciplined RL Training Notebook
### Closed-Loop DBS Agent for Parkinson's Disease

**Environment**: `virustechhacks/parkinsons_Motor` on Hugging Face Spaces  
**Target**: Train an agent that maps brain state → DBS parameters to maximise motor function  
**Pipeline**:
```
1. Connect to HF Space  →  explore environment
2. Evaluate baselines   →  establish floor/ceiling
3. Collect expert data  →  Fleming PID trajectories
4. Behavioral Cloning   →  warm-start policy
5. RL fine-tuning       →  PPO vs SAC vs TD3 comparison
6. Curriculum training  →  easy → medium → hard task ladder
7. Final evaluation     →  grader scores across all 6 tasks
```

---
**Observation space**: 27 continuous fields (beta_arv, tremor_arv, force_preserved, side_effect_load, ...)  
**Action space**: 4 continuous (dbs_amplitude [0-5 mA], dbs_pulse_width [0.06-0.20 ms], dbs_frequency [60-185 Hz], motor_command [-1,1])  
**Reward**: Dense, multi-objective — force preservation + beta suppression + safety budget  
**Tasks**: beta_suppression (easy/0.50), tremor_correction (medium/0.40), full_episode (hard/0.62)

---
## Section 0 — Install Dependencies

In [ ]:
# Core packages
!pip install -q openenv-core stable-baselines3[extra] gymnasium torch tensorboard
!pip install -q nest_asyncio pandas matplotlib seaborn tqdm

# Clone the HF Space repo to get ParkinsonsMotorEnv client + models
import os
SPACE_REPO = "https://huggingface.co/spaces/virustechhacks/parkinsons_Motor"
CLONE_DIR  = "/tmp/parkinsons_motor_space"

if not os.path.exists(CLONE_DIR):
    !git clone {SPACE_REPO} {CLONE_DIR}
else:
    print("[INFO] Repo already cloned — skipping")

# Add the cloned package to Python path
import sys
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
print("[OK] Dependencies installed")

In [ ]:
# Patch asyncio for Kaggle/Jupyter (prevents 'event loop already running' errors)
import nest_asyncio
nest_asyncio.apply()

# Standard imports
import asyncio
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any, Callable
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Gymnasium + Stable-Baselines3
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, SAC, TD3
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[OK] Imports done | Device: {DEVICE}")

---
## Section 1 — Environment Connection & Configuration

In [ ]:
# ── Environment URL ────────────────────────────────────────────────────────────
# Option A: HF Space (deployed, shared)
HF_SPACE_URL = "https://virustechhacks-parkinsons-motor.hf.space"

# Option B: Local server (if running locally)
# HF_SPACE_URL = "http://localhost:8000"

# ── Import client & types ──────────────────────────────────────────────────────
from parkinsons_Motor import ParkinsonsMotorEnv, ParkinsonsMotorAction, ParkinsonsMotorObservation

# ── Task definitions ───────────────────────────────────────────────────────────
TASKS = {
    "beta_suppression":              {"difficulty": "easy",   "steps": 30,  "threshold": 0.50},
    "tremor_correction":             {"difficulty": "medium", "steps": 48,  "threshold": 0.40},
    "full_episode":                  {"difficulty": "hard",   "steps": 100, "threshold": 0.62},
    "fragile_patient":               {"difficulty": "expert", "steps": 64,  "threshold": 0.44},
    "refractory_patient":            {"difficulty": "expert", "steps": 100, "threshold": 0.42},
    "personalization_generalization":{"difficulty": "expert", "steps": 72,  "threshold": 0.45},
}
CORE_TASKS    = ["beta_suppression", "tremor_correction", "full_episode"]
EXPERT_TASKS  = ["fragile_patient", "refractory_patient", "personalization_generalization"]
CURRICULUM    = CORE_TASKS  # train order: easy → medium → hard

# ── Observation field names (order matters — must match obs_to_array) ──────────
OBS_FIELDS = [
    # Neural biomarkers
    "beta_arv", "tremor_arv", "semg_arv", "gamma_arv",
    # Motor function
    "force_preserved", "tracking_accuracy", "task_error",
    "target_output", "effective_motor_output",
    # Disease summary
    "disease_severity", "beta_suppression",
    # Temporal trends
    "beta_trend", "tremor_trend", "side_effect_rate",
    # DBS device state
    "dbs_amplitude_ma", "dbs_pulse_width_ms", "dbs_entrainment",
    "recent_dbs_avg_ma", "recent_dbs_avg_pw_ms",
    "side_effect_load", "action_smoothness_cost", "dbs_constraint_violation",
    "stim_washout", "battery_drain_rate",
    # Patient physiology
    "medication_phase",
    # Timing
    "sim_time_s",
]
OBS_DIM = len(OBS_FIELDS)

# ── Action bounds ──────────────────────────────────────────────────────────────
ACT_LOW  = np.array([0.0,   0.06,  60.0, -1.0], dtype=np.float32)
ACT_HIGH = np.array([5.0,   0.20, 185.0,  1.0], dtype=np.float32)
ACT_DIM  = 4

print(f"[OK] Observation dim: {OBS_DIM} | Action dim: {ACT_DIM}")
print(f"[OK] Tasks: {list(TASKS.keys())}")

In [ ]:
# ── Conversion helpers ─────────────────────────────────────────────────────────

def obs_to_array(obs: ParkinsonsMotorObservation) -> np.ndarray:
    """Convert observation object → float32 numpy vector."""
    return np.array(
        [getattr(obs, f, 0.0) for f in OBS_FIELDS],
        dtype=np.float32
    )

def array_to_action(arr: np.ndarray, task_id: str = "") -> ParkinsonsMotorAction:
    """Convert float32 numpy vector → ParkinsonsMotorAction."""
    arr = np.clip(arr, ACT_LOW, ACT_HIGH)
    return ParkinsonsMotorAction(
        dbs_amplitude  = float(arr[0]),
        dbs_pulse_width= float(arr[1]),
        dbs_frequency  = float(arr[2]),
        motor_command  = float(arr[3]),
        task_id        = task_id,
    )

def safe_get(obs: ParkinsonsMotorObservation, field: str, default: float = 0.0) -> float:
    return float(getattr(obs, field, default))

# ── Quick connection test ──────────────────────────────────────────────────────
async def test_connection():
    env = ParkinsonsMotorEnv(base_url=HF_SPACE_URL)
    try:
        result = await env.reset(task_id="beta_suppression")
        obs = result.observation
        print(f"[OK] Connected to {HF_SPACE_URL}")
        print(f"     beta_arv={obs.beta_arv:.3f}  tremor_arv={obs.tremor_arv:.3f}")
        print(f"     force_preserved={obs.force_preserved:.3f}  task_id={obs.task_id}")
        return True
    except Exception as e:
        print(f"[FAIL] Connection error: {e}")
        print("       Check HF_SPACE_URL or start a local server")
        return False
    finally:
        await env.close()

asyncio.run(test_connection())

---
## Section 2 — Synchronous Gymnasium Wrapper

Wraps the async WebSocket client into a sync `gym.Env` compatible with Stable-Baselines3.

In [ ]:
class DBSGymEnv(gym.Env):
    """
    Synchronous Gymnasium wrapper around the async ParkinsonsMotorEnv client.
    Connects to the HF Space via WebSocket; runs async calls via asyncio event loop.
    """
    metadata = {"render_modes": []}

    def __init__(self, base_url: str, task_id: str = "beta_suppression", seed: int = 0):
        super().__init__()
        self.base_url  = base_url
        self.task_id   = task_id
        self._seed     = seed
        self._loop     = asyncio.get_event_loop()
        self._client   = ParkinsonsMotorEnv(base_url=base_url)
        self._episode_reward = 0.0
        self._step_count     = 0
        self._last_obs       = None

        # Gymnasium spaces
        self.observation_space = spaces.Box(
            low   = -np.inf,
            high  =  np.inf,
            shape = (OBS_DIM,),
            dtype = np.float32,
        )
        self.action_space = spaces.Box(
            low   = ACT_LOW,
            high  = ACT_HIGH,
            shape = (ACT_DIM,),
            dtype = np.float32,
        )

    def reset(self, seed=None, options=None):
        self._episode_reward = 0.0
        self._step_count     = 0
        result = self._loop.run_until_complete(
            self._client.reset(task_id=self.task_id)
        )
        self._last_obs = result.observation
        return obs_to_array(result.observation), {}

    def step(self, action: np.ndarray):
        act = array_to_action(action, task_id=self.task_id)
        result = self._loop.run_until_complete(self._client.step(act))
        obs  = result.observation
        rew  = float(result.reward) if result.reward is not None else 0.0
        done = bool(result.done)
        self._last_obs        = obs
        self._episode_reward += rew
        self._step_count     += 1
        info = {
            "grader_score":    safe_get(obs, "grader_score", -1.0),
            "episode_success": getattr(obs, "episode_success", False),
            "force_preserved": safe_get(obs, "force_preserved"),
            "beta_arv":        safe_get(obs, "beta_arv"),
            "side_effect_load":safe_get(obs, "side_effect_load"),
        }
        return obs_to_array(obs), rew, done, False, info

    def close(self):
        self._loop.run_until_complete(self._client.close())

    def __repr__(self):
        return f"DBSGymEnv(task={self.task_id}, url={self.base_url})"


def make_env(task_id: str, base_url: str = HF_SPACE_URL) -> Callable:
    """Factory function for DummyVecEnv."""
    def _init():
        env = DBSGymEnv(base_url=base_url, task_id=task_id)
        env = Monitor(env)
        return env
    return _init


# Quick sanity check
print("[TEST] Instantiating gym wrapper...")
_test_env = DBSGymEnv(base_url=HF_SPACE_URL, task_id="beta_suppression")
obs, _ = _test_env.reset()
print(f"  obs shape: {obs.shape} | dtype: {obs.dtype}")
print(f"  action space: {_test_env.action_space}")
rand_act = _test_env.action_space.sample()
obs2, rew, done, _, info = _test_env.step(rand_act)
print(f"  step ok | reward={rew:.4f} | done={done} | force={info['force_preserved']:.3f}")
_test_env.close()
print("[OK] Gym wrapper working")

---
## Section 3 — Baseline Policy Evaluation

Establishing the score floor and ceiling before any training.

In [ ]:
# ── Policy definitions ─────────────────────────────────────────────────────────

def policy_zero_stim(obs: np.ndarray) -> np.ndarray:
    """Zero DBS, zero motor command — worst case baseline."""
    return np.array([0.0, 0.06, 130.0, 0.0], dtype=np.float32)


def policy_constant(amp=1.0, pw=0.13, freq=130.0) -> Callable:
    """Constant DBS parameters, motor_command tracks target_output."""
    def _policy(obs: np.ndarray) -> np.ndarray:
        target = obs[OBS_FIELDS.index("target_output")]
        return np.array([amp, pw, freq, target], dtype=np.float32)
    return _policy


def policy_rule_based(obs: np.ndarray) -> np.ndarray:
    """
    Hand-crafted safety-aware policy:
    - Start moderate (1.0 mA)
    - Scale up with disease severity
    - Pull back when side_effect_load > 0.35
    - Always track target_output with motor_command
    """
    beta       = obs[OBS_FIELDS.index("beta_arv")]
    tremor     = obs[OBS_FIELDS.index("tremor_arv")]
    se_load    = obs[OBS_FIELDS.index("side_effect_load")]
    se_rate    = obs[OBS_FIELDS.index("side_effect_rate")]
    target     = obs[OBS_FIELDS.index("target_output")]
    disease    = obs[OBS_FIELDS.index("disease_severity")]

    # Amplitude: proportional to disease, safety-capped
    base_amp = 0.4 + 1.4 * disease
    if se_load > 0.40:
        base_amp *= max(0.3, 1.0 - (se_load - 0.40) * 3.0)
    if se_rate > 0.02:
        base_amp *= 0.85

    # Pulse width: wider when tremor is high
    pw = 0.09 + 0.06 * tremor

    # Frequency: standard 130 Hz, drop to 80 Hz for pure tremor
    freq = 80.0 if (tremor > 0.6 and beta < 0.3) else 130.0

    amp = np.clip(base_amp, 0.0, 2.4)
    pw  = np.clip(pw,  0.06, 0.20)
    return np.array([amp, pw, freq, target], dtype=np.float32)


BASELINES = {
    "zero_stim":    policy_zero_stim,
    "constant_1mA": policy_constant(amp=1.0, pw=0.13, freq=130.0),
    "constant_max": policy_constant(amp=2.4, pw=0.20, freq=130.0),
    "rule_based":   policy_rule_based,
}

print("[OK] Baseline policies defined")
print(f"     Policies: {list(BASELINES.keys())}")

In [ ]:
# ── Evaluation runner ──────────────────────────────────────────────────────────

@dataclass
class EpisodeResult:
    policy_name:    str
    task_id:        str
    grader_score:   float
    episode_success:bool
    total_reward:   float
    mean_force:     float
    mean_beta:      float
    mean_se_load:   float
    n_steps:        int


def run_episode(
    policy: Callable,
    task_id: str,
    base_url: str = HF_SPACE_URL,
    policy_name: str = "policy",
) -> EpisodeResult:
    env = DBSGymEnv(base_url=base_url, task_id=task_id)
    obs, _ = env.reset()

    total_reward = 0.0
    force_hist, beta_hist, se_hist = [], [], []
    final_score, success, n_steps = -1.0, False, 0

    done = False
    while not done:
        action = policy(obs)
        obs, rew, done, _, info = env.step(action)
        total_reward += rew
        force_hist.append(info["force_preserved"])
        beta_hist.append(info["beta_arv"])
        se_hist.append(info["side_effect_load"])
        n_steps += 1
        if done:
            final_score = info["grader_score"]
            success     = info["episode_success"]

    env.close()
    return EpisodeResult(
        policy_name     = policy_name,
        task_id         = task_id,
        grader_score    = final_score,
        episode_success = success,
        total_reward    = total_reward,
        mean_force      = float(np.mean(force_hist)) if force_hist else 0.0,
        mean_beta       = float(np.mean(beta_hist))  if beta_hist  else 0.0,
        mean_se_load    = float(np.mean(se_hist))    if se_hist    else 0.0,
        n_steps         = n_steps,
    )


def evaluate_policy_batch(
    policy: Callable,
    task_ids: List[str],
    n_episodes: int = 3,
    policy_name: str = "policy",
    base_url: str = HF_SPACE_URL,
) -> List[EpisodeResult]:
    results = []
    for task_id in task_ids:
        threshold = TASKS[task_id]["threshold"]
        for ep in range(n_episodes):
            r = run_episode(policy, task_id, base_url, policy_name)
            results.append(r)
            status = "PASS" if r.grader_score >= threshold else "FAIL"
            print(f"  [{status}] {policy_name:15s} | {task_id:30s} | ep{ep+1} "
                  f"score={r.grader_score:.3f} (need {threshold}) "
                  f"force={r.mean_force:.2f} se={r.mean_se_load:.2f}")
    return results


print("[OK] Evaluation runner ready")

In [ ]:
# ── Run all baselines on core tasks ───────────────────────────────────────────
# NOTE: Each episode takes ~2-5 seconds over WebSocket
# Adjust N_BASELINE_EPS to trade speed vs statistical reliability
N_BASELINE_EPS = 2  # increase to 5 for final benchmarking

all_baseline_results: List[EpisodeResult] = []

for name, policy in BASELINES.items():
    print(f"\n=== {name.upper()} ===")
    results = evaluate_policy_batch(
        policy      = policy,
        task_ids    = CORE_TASKS,
        n_episodes  = N_BASELINE_EPS,
        policy_name = name,
    )
    all_baseline_results.extend(results)

print("\n[OK] Baseline evaluation complete")

In [ ]:
# ── Baseline results table ─────────────────────────────────────────────────────
baseline_df = pd.DataFrame([
    {
        "Policy":    r.policy_name,
        "Task":      r.task_id,
        "Score":     r.grader_score,
        "Success":   r.episode_success,
        "Reward":    r.total_reward,
        "Force":     r.mean_force,
        "Beta":      r.mean_beta,
        "SE Load":   r.mean_se_load,
    }
    for r in all_baseline_results
])

summary_df = (
    baseline_df
    .groupby(["Policy", "Task"])[["Score", "Force", "Beta", "SE Load"]]
    .mean()
    .round(3)
)
print("\n=== BASELINE SUMMARY (mean over episodes) ===")
print(summary_df.to_string())

# Visual heatmap
fig, ax = plt.subplots(figsize=(10, 4))
pivot = baseline_df.groupby(["Policy", "Task"])["Score"].mean().unstack()
thresholds = [TASKS[t]["threshold"] for t in pivot.columns]
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=0.8,
            ax=ax, linewidths=0.5)
ax.set_title("Baseline Grader Scores (thresholds: 0.50 / 0.40 / 0.62)", fontsize=13)
plt.tight_layout()
plt.savefig("baseline_heatmap.png", dpi=150)
plt.show()

---
## Section 4 — Expert Data Collection (Fleming PID Trajectories)

The ground-truth Fleming PID controller data is embedded in the environment metadata.
We collect N episodes per task and extract (obs, action) pairs for behavioral cloning.

In [ ]:
@dataclass
class Transition:
    obs:        np.ndarray
    action:     np.ndarray
    reward:     float
    next_obs:   np.ndarray
    done:       bool
    task_id:    str
    grader_score: float = -1.0


def collect_expert_episode(
    task_id: str,
    base_url: str = HF_SPACE_URL,
    expert_policy: Callable = policy_rule_based,
) -> Tuple[List[Transition], float]:
    """
    Run one episode with the expert policy and collect transitions.
    Returns (transitions, grader_score).
    
    In a production setup, replace expert_policy with a loader that
    replays the Fleming PID actions from obs.metadata['ground_truth_amp'].
    """
    env = DBSGymEnv(base_url=base_url, task_id=task_id)
    obs, _ = env.reset()
    transitions = []
    done = False
    final_score = -1.0

    while not done:
        action = expert_policy(obs)
        next_obs, rew, done, _, info = env.step(action)
        transitions.append(Transition(
            obs        = obs.copy(),
            action     = action.copy(),
            reward     = rew,
            next_obs   = next_obs.copy(),
            done       = done,
            task_id    = task_id,
            grader_score = info["grader_score"] if done else -1.0,
        ))
        obs = next_obs
        if done:
            final_score = info["grader_score"]

    env.close()
    return transitions, final_score


def build_expert_dataset(
    task_ids:      List[str],
    n_episodes:    int = 10,
    expert_policy: Callable = policy_rule_based,
    score_filter:  float = 0.0,  # only keep episodes with score > threshold
) -> List[Transition]:
    """Collect expert transitions across tasks, optionally filtering by quality."""
    dataset: List[Transition] = []
    for task_id in task_ids:
        print(f"  Collecting {n_episodes} episodes for {task_id}...")
        kept = 0
        for ep in range(n_episodes):
            transitions, score = collect_expert_episode(task_id, HF_SPACE_URL, expert_policy)
            if score >= score_filter or score < 0:
                dataset.extend(transitions)
                kept += 1
            print(f"    ep{ep+1:02d} score={score:.3f} {'[kept]' if score >= score_filter or score < 0 else '[dropped]'}")
        print(f"  Task {task_id}: kept {kept}/{n_episodes} episodes, {len(dataset)} transitions total")
    return dataset


print("[OK] Expert data collection ready")
print("     Running on rule_based as proxy for Fleming PID expert")
print("     Set N_EXPERT_EPS to control dataset size")

In [ ]:
# ── Collect expert demonstrations ──────────────────────────────────────────────
N_EXPERT_EPS = 5   # episodes per task; increase to 20+ for production BC

print("=== EXPERT DATA COLLECTION ===")
expert_dataset = build_expert_dataset(
    task_ids      = CORE_TASKS,
    n_episodes    = N_EXPERT_EPS,
    expert_policy = policy_rule_based,
    score_filter  = 0.30,
)

print(f"\n[OK] Expert dataset: {len(expert_dataset)} transitions")
print(f"     Obs shape per step: {expert_dataset[0].obs.shape}")
print(f"     Action shape per step: {expert_dataset[0].action.shape}")

# Dataset stats
actions = np.stack([t.action for t in expert_dataset])
print(f"\nAction stats (expert dataset):")
print(f"  dbs_amplitude  mean={actions[:,0].mean():.3f}  std={actions[:,0].std():.3f}  range=[{actions[:,0].min():.2f}, {actions[:,0].max():.2f}]")
print(f"  dbs_pulse_width mean={actions[:,1].mean():.4f} std={actions[:,1].std():.4f}")
print(f"  dbs_frequency   mean={actions[:,2].mean():.1f}  std={actions[:,2].std():.2f}")
print(f"  motor_command   mean={actions[:,3].mean():.3f}  std={actions[:,3].std():.3f}")

---
## Section 5 — Behavioral Cloning (SFT Warm-Start)

Train a policy network via supervised learning on expert trajectories.
This warm-starts RL training into a clinically safe parameter region,
avoiding the cold-start exploration trap in the entrainment dead zone.

In [ ]:
# ── Network architecture ───────────────────────────────────────────────────────

class BCPolicy(nn.Module):
    """
    MLP policy for behavioral cloning.
    Input:  OBS_DIM observation vector
    Output: ACT_DIM continuous actions (pre-sigmoid, scaled to action bounds)
    """
    def __init__(self, obs_dim: int = OBS_DIM, act_dim: int = ACT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, act_dim),
            nn.Sigmoid(),  # output in [0, 1], scaled below
        )
        # Action scaling parameters (registered as buffers for device compatibility)
        act_low  = torch.tensor(ACT_LOW,  dtype=torch.float32)
        act_high = torch.tensor(ACT_HIGH, dtype=torch.float32)
        self.register_buffer("act_low",   act_low)
        self.register_buffer("act_range", act_high - act_low)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        raw = self.net(obs)
        return self.act_low + raw * self.act_range

    @torch.no_grad()
    def predict(self, obs: np.ndarray) -> np.ndarray:
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.act_low.device)
        return self.forward(obs_t).squeeze(0).cpu().numpy()


model_bc = BCPolicy().to(DEVICE)
print(f"[OK] BCPolicy | params: {sum(p.numel() for p in model_bc.parameters()):,}")
print(f"     Architecture: {OBS_DIM} → 256 → 256 → 128 → {ACT_DIM}")

In [ ]:
# ── Training setup ─────────────────────────────────────────────────────────────

def prepare_bc_dataset(
    transitions: List[Transition],
    val_split: float = 0.15,
    batch_size: int = 256,
) -> Tuple[DataLoader, DataLoader]:
    obs_np  = np.stack([t.obs    for t in transitions]).astype(np.float32)
    act_np  = np.stack([t.action for t in transitions]).astype(np.float32)

    obs_t  = torch.tensor(obs_np)
    act_t  = torch.tensor(act_np)

    n      = len(obs_t)
    n_val  = int(n * val_split)
    idx    = torch.randperm(n)
    val_idx, train_idx = idx[:n_val], idx[n_val:]

    train_loader = DataLoader(
        TensorDataset(obs_t[train_idx], act_t[train_idx]),
        batch_size=batch_size, shuffle=True, drop_last=True,
    )
    val_loader = DataLoader(
        TensorDataset(obs_t[val_idx], act_t[val_idx]),
        batch_size=batch_size, shuffle=False,
    )
    print(f"  Train: {len(train_idx)} samples | Val: {len(val_idx)} samples")
    return train_loader, val_loader


def train_bc(
    model:         BCPolicy,
    train_loader:  DataLoader,
    val_loader:    DataLoader,
    n_epochs:      int   = 40,
    lr:            float = 3e-4,
    patience:      int   = 8,
    save_path:     str   = "bc_policy.pt",
) -> Dict[str, List[float]]:

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.MSELoss()

    history = {"train_loss": [], "val_loss": []}
    best_val, patience_counter = float('inf'), 0

    pbar = tqdm(range(n_epochs), desc="BC Training")
    for epoch in pbar:
        # Train
        model.train()
        train_losses = []
        for obs_b, act_b in train_loader:
            obs_b, act_b = obs_b.to(DEVICE), act_b.to(DEVICE)
            pred = model(obs_b)
            loss = criterion(pred, act_b)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for obs_b, act_b in val_loader:
                obs_b, act_b = obs_b.to(DEVICE), act_b.to(DEVICE)
                val_losses.append(criterion(model(obs_b), act_b).item())

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        scheduler.step()

        pbar.set_postfix({"train": f"{train_loss:.5f}", "val": f"{val_loss:.5f}"})

        if val_loss < best_val:
            best_val = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stop at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    print(f"  [OK] Best val loss: {best_val:.6f}")
    return history


# Build dataset and train
print("=== BEHAVIORAL CLONING ===")
train_loader, val_loader = prepare_bc_dataset(expert_dataset)
bc_history = train_bc(
    model_bc, train_loader, val_loader,
    n_epochs=50, lr=3e-4, patience=10, save_path="bc_policy.pt",
)

In [ ]:
# ── BC training curves ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bc_history["train_loss"], label="Train MSE", lw=2)
ax.plot(bc_history["val_loss"],   label="Val MSE",   lw=2, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Behavioral Cloning — Training Curve")
ax.legend()
plt.tight_layout()
plt.savefig("bc_training_curve.png", dpi=150)
plt.show()

# ── BC policy evaluation ───────────────────────────────────────────────────────
model_bc.eval()
bc_policy = lambda obs: model_bc.predict(obs)

print("\n=== BC POLICY EVALUATION ===")
bc_results = evaluate_policy_batch(
    policy      = bc_policy,
    task_ids    = CORE_TASKS,
    n_episodes  = N_BASELINE_EPS,
    policy_name = "bc_policy",
)
bc_df = pd.DataFrame([vars(r) for r in bc_results])
print("\nBC grader scores:")
print(bc_df.groupby("task_id")["grader_score"].mean().round(3))

---
## Section 6 — RL Algorithm Comparison: PPO vs SAC vs TD3

Train each algorithm from **BC initialization** on the easy task first.  
Compare sample efficiency and final grader score.

In [ ]:
# ── SB3 policy initialization from BC weights ─────────────────────────────────

def load_bc_weights_into_sb3(sb3_model, bc_model: BCPolicy) -> None:
    """
    Copy BC network weights into the SB3 actor/policy network.
    Only copies layers with matching dimensions (first 3 linear layers).
    """
    bc_state  = bc_model.state_dict()
    sb3_state = sb3_model.policy.state_dict()

    # SB3 MlpPolicy actor prefix varies by algorithm
    copied = 0
    for sb3_key in sb3_state:
        if "mlp_extractor" in sb3_key or "action_net" in sb3_key or "actor" in sb3_key:
            for bc_key in bc_state:
                if sb3_state[sb3_key].shape == bc_state[bc_key].shape:
                    sb3_state[sb3_key] = bc_state[bc_key].clone()
                    copied += 1
                    break
    sb3_model.policy.load_state_dict(sb3_state, strict=False)
    print(f"  [BC init] Copied {copied} parameter tensors into SB3 policy")


# ── Training configuration ─────────────────────────────────────────────────────

# How many env steps to train — increase for real training
# Recommendation: PPO 50k, SAC 30k, TD3 30k per task stage
TRAIN_STEPS_EASY   = 10_000   # beta_suppression (30 steps/ep ~ 333 episodes)
TRAIN_STEPS_MEDIUM = 15_000   # tremor_correction
TRAIN_STEPS_HARD   = 25_000   # full_episode

# SB3 hyperparameters
PPO_KWARGS = dict(
    policy          = "MlpPolicy",
    learning_rate   = 3e-4,
    n_steps         = 512,
    batch_size      = 64,
    n_epochs        = 10,
    gamma           = 0.99,
    gae_lambda      = 0.95,
    clip_range      = 0.2,
    ent_coef        = 0.01,
    policy_kwargs   = dict(net_arch=[256, 256]),
    verbose         = 0,
)

SAC_KWARGS = dict(
    policy          = "MlpPolicy",
    learning_rate   = 3e-4,
    buffer_size     = 50_000,
    batch_size      = 256,
    tau             = 0.005,
    gamma           = 0.99,
    train_freq      = 1,
    gradient_steps  = 1,
    ent_coef        = "auto",
    target_entropy  = "auto",
    policy_kwargs   = dict(net_arch=[256, 256]),
    verbose         = 0,
)

TD3_KWARGS = dict(
    policy          = "MlpPolicy",
    learning_rate   = 3e-4,
    buffer_size     = 50_000,
    batch_size      = 256,
    tau             = 0.005,
    gamma           = 0.99,
    train_freq      = (1, "episode"),
    action_noise    = None,
    policy_kwargs   = dict(net_arch=[256, 256]),
    verbose         = 0,
)

print("[OK] Training configurations set")
print(f"     Steps: easy={TRAIN_STEPS_EASY:,} | medium={TRAIN_STEPS_MEDIUM:,} | hard={TRAIN_STEPS_HARD:,}")

In [ ]:
# ── Grader-score tracking callback ────────────────────────────────────────────

class GraderScoreCallback(BaseCallback):
    """
    Logs grader_score and episode_success at each episode end.
    Stores rolling mean for plotting.
    """
    def __init__(self, task_id: str, window: int = 10, verbose=0):
        super().__init__(verbose)
        self.task_id       = task_id
        self.window        = window
        self.scores:  List[float] = []
        self.rewards: List[float] = []
        self.successes: List[bool] = []
        self._ep_reward    = 0.0

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            if info.get("grader_score", -1.0) >= 0:
                self.scores.append(info["grader_score"])
                self.successes.append(info.get("episode_success", False))
        rewards = self.locals.get("rewards", [])
        for r in rewards:
            self.rewards.append(float(r))
        return True

    def mean_score(self) -> float:
        if not self.scores:
            return 0.0
        return float(np.mean(self.scores[-self.window:]))

    def success_rate(self) -> float:
        if not self.successes:
            return 0.0
        return float(np.mean(self.successes[-self.window:]))


def train_algorithm(
    algo_class,
    algo_kwargs:   dict,
    task_id:       str,
    total_steps:   int,
    algo_name:     str = "algo",
    bc_warmstart:  bool = True,
    base_url:      str = HF_SPACE_URL,
) -> Tuple[Any, GraderScoreCallback]:
    """
    Train one RL algorithm on one task from scratch (or BC warm-start).
    Returns (trained_model, callback).
    """
    print(f"\n--- {algo_name.upper()} on {task_id} ({total_steps:,} steps) ---")

    # Single environment (no vec normalization for now — env already normalized)
    env = DummyVecEnv([make_env(task_id, base_url)])

    model    = algo_class(env=env, **algo_kwargs)
    callback = GraderScoreCallback(task_id=task_id)

    if bc_warmstart:
        try:
            load_bc_weights_into_sb3(model, model_bc)
        except Exception as e:
            print(f"  [WARN] BC warm-start failed: {e} — training from scratch")

    t0 = time.time()
    model.learn(total_timesteps=total_steps, callback=callback, progress_bar=True)
    elapsed = time.time() - t0

    print(f"  Done in {elapsed:.0f}s | mean_score={callback.mean_score():.3f} "
          f"| success_rate={callback.success_rate():.1%}")

    model.save(f"{algo_name}_{task_id}.zip")
    env.close()
    return model, callback


print("[OK] Training utilities ready")

In [ ]:
# ── Algorithm comparison on beta_suppression (easy task) ──────────────────────
# Run all three algorithms; compare learning speed and final score

comparison_results: Dict[str, Dict] = {}

ALGORITHMS = {
    "PPO": (PPO, PPO_KWARGS),
    "SAC": (SAC, SAC_KWARGS),
    "TD3": (TD3, TD3_KWARGS),
}

trained_models: Dict[str, Any] = {}

for algo_name, (algo_class, algo_kwargs) in ALGORITHMS.items():
    model, cb = train_algorithm(
        algo_class   = algo_class,
        algo_kwargs  = algo_kwargs,
        task_id      = "beta_suppression",
        total_steps  = TRAIN_STEPS_EASY,
        algo_name    = algo_name,
        bc_warmstart = True,
    )
    trained_models[algo_name]    = model
    comparison_results[algo_name] = {"callback": cb, "model": model}

print("\n[OK] Algorithm comparison training complete")

In [ ]:
# ── Learning curve comparison plot ────────────────────────────────────────────

def smooth(values: List[float], window: int = 10) -> np.ndarray:
    if len(values) < window:
        return np.array(values)
    return np.convolve(values, np.ones(window)/window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {"PPO": "#2196F3", "SAC": "#4CAF50", "TD3": "#FF5722"}

for algo_name, res in comparison_results.items():
    cb = res["callback"]
    scores = cb.scores
    if scores:
        s = smooth(scores, window=5)
        axes[0].plot(s, label=algo_name, color=colors[algo_name], lw=2)

axes[0].axhline(TASKS["beta_suppression"]["threshold"], color="red",
                linestyle="--", lw=1.5, label="Success threshold (0.50)")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Grader Score")
axes[0].set_title("beta_suppression — Learning Curves")
axes[0].legend()
axes[0].set_ylim(0, 1)

# Final score bar chart
final_scores = {
    algo: res["callback"].mean_score()
    for algo, res in comparison_results.items()
}
bars = axes[1].bar(final_scores.keys(), final_scores.values(),
                   color=[colors[a] for a in final_scores])
axes[1].axhline(TASKS["beta_suppression"]["threshold"], color="red",
                linestyle="--", lw=1.5, label="Threshold")
for bar, val in zip(bars, final_scores.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel("Mean Score (last 10 eps)")
axes[1].set_title("Final Score Comparison")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("algorithm_comparison.png", dpi=150)
plt.show()

# Select best algorithm for curriculum training
best_algo = max(final_scores, key=final_scores.get)
print(f"\n[BEST] Algorithm for curriculum: {best_algo} (score={final_scores[best_algo]:.3f})")

---
## Section 7 — Curriculum Training

Progressive task ladder: easy → medium → hard.  
Each stage transfers weights from the previous stage.

In [ ]:
# ── Curriculum training loop ───────────────────────────────────────────────────

CURRICULUM_CONFIG = [
    {"task_id": "beta_suppression",  "steps": TRAIN_STEPS_EASY,   "threshold": 0.50},
    {"task_id": "tremor_correction", "steps": TRAIN_STEPS_MEDIUM, "threshold": 0.40},
    {"task_id": "full_episode",      "steps": TRAIN_STEPS_HARD,   "threshold": 0.62},
]

# Start from best algo trained above; continue training
AlgoClass, algo_kwargs = ALGORITHMS[best_algo]

curriculum_callbacks: Dict[str, GraderScoreCallback] = {}
current_model = trained_models[best_algo]  # warm-started from easy task

print(f"=== CURRICULUM TRAINING ({best_algo}) ===")

for stage in CURRICULUM_CONFIG:
    task_id   = stage["task_id"]
    steps     = stage["steps"]
    threshold = stage["threshold"]

    print(f"\n[STAGE] {task_id} | {steps:,} steps | threshold={threshold}")

    # Create new environment for this task
    env = DummyVecEnv([make_env(task_id)])

    # Transfer weights: set environment on existing model
    current_model.set_env(env)

    cb = GraderScoreCallback(task_id=task_id)
    current_model.learn(
        total_timesteps  = steps,
        callback         = cb,
        reset_num_timesteps = False,  # continue step counter
        progress_bar     = True,
    )

    mean_score = cb.mean_score()
    success_rate = cb.success_rate()
    status = "PASS" if mean_score >= threshold else "BELOW"
    print(f"  [{status}] mean_score={mean_score:.3f} | success_rate={success_rate:.1%}")

    curriculum_callbacks[task_id] = cb
    current_model.save(f"curriculum_{best_algo}_{task_id}.zip")
    env.close()

print("\n[OK] Curriculum training complete")
best_model = current_model

In [ ]:
# ── Curriculum learning curves ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
stage_colors = {"beta_suppression": "#2196F3", "tremor_correction": "#FF9800", "full_episode": "#9C27B0"}

for ax, (task_id, cb) in zip(axes, curriculum_callbacks.items()):
    color     = stage_colors.get(task_id, "gray")
    threshold = TASKS[task_id]["threshold"]
    scores    = cb.scores

    if scores:
        s = smooth(scores, window=5)
        ax.plot(s, color=color, lw=2, label=f"{best_algo}")
        ax.fill_between(range(len(s)), s, alpha=0.15, color=color)

    ax.axhline(threshold, color="red", linestyle="--", lw=1.5,
               label=f"Threshold ({threshold})")
    ax.set_title(f"{task_id}\n({TASKS[task_id]['difficulty']})", fontsize=11)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Grader Score")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

plt.suptitle(f"Curriculum Training — {best_algo} with BC Warm-Start", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("curriculum_learning_curves.png", dpi=150)
plt.show()

---
## Section 8 — Final Evaluation

Evaluate the best curriculum-trained model against all baselines on all 6 tasks.

In [ ]:
# ── Wrap trained SB3 model as a callable policy ────────────────────────────────

def sb3_policy(model) -> Callable:
    """Convert an SB3 model to a callable policy function."""
    def _policy(obs: np.ndarray) -> np.ndarray:
        action, _ = model.predict(obs, deterministic=True)
        return action.astype(np.float32)
    return _policy

# All policies to evaluate in final benchmark
ALL_TASKS = CORE_TASKS + EXPERT_TASKS
N_EVAL_EPS = 3  # increase to 5 for final publication

final_policies = {
    "zero_stim":     policy_zero_stim,
    "constant_1mA":  policy_constant(amp=1.0, pw=0.13, freq=130.0),
    "rule_based":    policy_rule_based,
    "bc_policy":     bc_policy,
    f"{best_algo}_curriculum": sb3_policy(best_model),
}

# Also add per-algo comparison models on easy task
for algo_name, res in comparison_results.items():
    final_policies[f"{algo_name}_easy"] = sb3_policy(res["model"])

print(f"[OK] Final evaluation setup")
print(f"     Policies: {list(final_policies.keys())}")
print(f"     Tasks:    {ALL_TASKS}")
print(f"     Episodes per combination: {N_EVAL_EPS}")

In [ ]:
# ── Run final evaluation ───────────────────────────────────────────────────────
final_results: List[EpisodeResult] = []

print("=== FINAL BENCHMARK EVALUATION ===")
for policy_name, policy in final_policies.items():
    print(f"\n--- {policy_name} ---")
    results = evaluate_policy_batch(
        policy      = policy,
        task_ids    = ALL_TASKS,
        n_episodes  = N_EVAL_EPS,
        policy_name = policy_name,
    )
    final_results.extend(results)

print("\n[OK] Final evaluation complete")

In [ ]:
# ── Results analysis ───────────────────────────────────────────────────────────

results_df = pd.DataFrame([
    {
        "Policy":    r.policy_name,
        "Task":      r.task_id,
        "Score":     r.grader_score,
        "Success":   int(r.episode_success),
        "Reward":    r.total_reward,
        "Force":     r.mean_force,
        "Beta":      r.mean_beta,
        "SE Load":   r.mean_se_load,
        "Steps":     r.n_steps,
    }
    for r in final_results
])

# Mean across episodes
pivot_score = (
    results_df
    .groupby(["Policy", "Task"])["Score"]
    .mean()
    .unstack()
    .round(3)
)

print("\n=== FINAL GRADER SCORE MATRIX ===")
print(pivot_score.to_string())

# Success rate matrix
pivot_success = (
    results_df
    .groupby(["Policy", "Task"])["Success"]
    .mean()
    .unstack()
    .round(2)
)
print("\n=== SUCCESS RATE MATRIX ===")
print(pivot_success.to_string())

In [ ]:
# ── Comprehensive results visualization ────────────────────────────────────────

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# 1. Score heatmap (all policies × all tasks)
ax1 = fig.add_subplot(gs[0, :])
threshold_vals = [TASKS[t]["threshold"] for t in pivot_score.columns]
annot = pivot_score.copy()
sns.heatmap(
    pivot_score, annot=annot, fmt=".3f",
    cmap="RdYlGn", vmin=0.0, vmax=0.80,
    ax=ax1, linewidths=0.5, cbar_kws={"shrink": 0.8},
)
# Add threshold markers
for i, threshold in enumerate(threshold_vals):
    ax1.text(i + 0.5, len(pivot_score) + 0.3,
             f"t={threshold}", ha='center', fontsize=8, color='red')
ax1.set_title("Final Benchmark — Mean Grader Score (red = success threshold)",
              fontsize=13, pad=20)
ax1.set_xlabel("Task", fontsize=11)
ax1.set_ylabel("Policy", fontsize=11)
plt.setp(ax1.get_xticklabels(), rotation=20, ha='right')

# 2. Core tasks bar chart
ax2 = fig.add_subplot(gs[1, 0])
core_pivot = pivot_score[CORE_TASKS] if CORE_TASKS[0] in pivot_score.columns else pivot_score
core_pivot.plot(kind="bar", ax=ax2, colormap="Set2", edgecolor="white", lw=0.5)
for i, (task, threshold) in enumerate(zip(CORE_TASKS, [0.50, 0.40, 0.62])):
    ax2.axhline(threshold, linestyle="--", lw=1, alpha=0.7,
                color=plt.cm.Set2(i/3), label=f"_")
ax2.set_title("Core Tasks — Score by Policy", fontsize=11)
ax2.set_xlabel("")
ax2.set_ylabel("Mean Grader Score")
ax2.set_ylim(0, 1)
ax2.legend(title="Task", fontsize=8, bbox_to_anchor=(1.0, 1.0))
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', fontsize=8)

# 3. Safety vs Force trade-off scatter
ax3 = fig.add_subplot(gs[1, 1])
mean_by_policy = results_df.groupby("Policy")[["Force", "SE Load", "Score"]].mean()
scatter = ax3.scatter(
    mean_by_policy["SE Load"], mean_by_policy["Force"],
    c=mean_by_policy["Score"], cmap="RdYlGn",
    s=200, edgecolors="k", lw=1, vmin=0, vmax=0.8,
)
for policy, row in mean_by_policy.iterrows():
    ax3.annotate(policy, (row["SE Load"], row["Force"]),
                 textcoords="offset points", xytext=(6, 4), fontsize=8)
plt.colorbar(scatter, ax=ax3, label="Mean Score")
ax3.set_xlabel("Mean Side-Effect Load")
ax3.set_ylabel("Mean Force Preserved")
ax3.set_title("Safety vs Motor Function Trade-off", fontsize=11)

plt.suptitle("MotorAssistEnv — Full Benchmark Results", fontsize=15, y=1.01, fontweight='bold')
plt.savefig("full_benchmark_results.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Section 9 — Deep-Dive: Best Policy Trajectory Analysis

In [ ]:
# ── Collect a full trajectory for detailed analysis ────────────────────────────

def collect_trajectory(
    policy: Callable,
    task_id: str = "full_episode",
    base_url: str = HF_SPACE_URL,
) -> pd.DataFrame:
    """Run one episode and return a DataFrame with every step's state."""
    env = DBSGymEnv(base_url=base_url, task_id=task_id)
    obs, _ = env.reset()
    records = []
    done = False
    step = 0

    while not done:
        action = policy(obs)
        next_obs, rew, done, _, info = env.step(action)
        record = {f: obs[OBS_FIELDS.index(f)] for f in OBS_FIELDS}
        record.update({
            "step":          step,
            "reward":        rew,
            "act_amp":       float(action[0]),
            "act_pw":        float(action[1]),
            "act_freq":      float(action[2]),
            "act_motor":     float(action[3]),
            "grader_score":  info["grader_score"],
        })
        records.append(record)
        obs = next_obs
        step += 1

    env.close()
    return pd.DataFrame(records)


print("Collecting trajectory for best model on full_episode...")
best_policy_fn = sb3_policy(best_model)
traj_best  = collect_trajectory(best_policy_fn,  task_id="full_episode")
traj_rule  = collect_trajectory(policy_rule_based, task_id="full_episode")
traj_const = collect_trajectory(policy_constant(amp=1.0, pw=0.13, freq=130.0), task_id="full_episode")
print(f"[OK] Trajectories collected: {len(traj_best)} steps each")

In [ ]:
# ── Trajectory plot ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)

trajs = [
    (traj_const, "constant_1mA",   "#9E9E9E", "--"),
    (traj_rule,  "rule_based",     "#FF9800", "-."),
    (traj_best,  f"{best_algo}_curriculum", "#4CAF50", "-"),
]

for t, label, color, ls in trajs:
    axes[0].plot(t["beta_arv"],       color=color, lw=2, ls=ls, label=label)
    axes[0].plot(t["tremor_arv"],     color=color, lw=1, ls=ls, alpha=0.5)
    axes[1].plot(t["force_preserved"],color=color, lw=2, ls=ls, label=label)
    axes[2].plot(t["act_amp"],        color=color, lw=2, ls=ls, label=label)
    axes[3].plot(t["side_effect_load"],color=color, lw=2, ls=ls, label=label)

# Reference lines
axes[0].axhline(0.26, color='k', lw=1, ls=':', alpha=0.5, label="beta target")
axes[1].axhline(0.60, color='k', lw=1, ls=':', alpha=0.5, label="force target")
axes[3].axhline(0.65, color='r', lw=1, ls=':', alpha=0.8, label="SE budget")

labels_y = ["Beta / Tremor ARV", "Force Preserved", "DBS Amplitude (mA)", "Side-Effect Load"]
for ax, ylabel in zip(axes, labels_y):
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(bottom=0)

axes[-1].set_xlabel("Step (20ms each)", fontsize=11)
plt.suptitle("full_episode Trajectory Comparison — 100 Steps", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("trajectory_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Section 10 — Summary & What's Next

In [ ]:
# ── Final summary table ────────────────────────────────────────────────────────

def summarize_results(results_df: pd.DataFrame) -> pd.DataFrame:
    summary = []
    for (policy, task), group in results_df.groupby(["Policy", "Task"]):
        threshold = TASKS.get(task, {}).get("threshold", 0.5)
        mean_score = group["Score"].mean()
        summary.append({
            "Policy":        policy,
            "Task":          task,
            "Difficulty":    TASKS.get(task, {}).get("difficulty", "?"),
            "Mean Score":    round(mean_score, 3),
            "Threshold":     threshold,
            "Passes":        mean_score >= threshold,
            "Success Rate":  round(group["Success"].mean(), 2),
            "Mean Force":    round(group["Force"].mean(), 3),
            "Mean SE Load":  round(group["SE Load"].mean(), 3),
        })
    return pd.DataFrame(summary).sort_values(["Difficulty", "Policy"])


summary = summarize_results(results_df)
print("\n" + "="*80)
print("FINAL BENCHMARK SUMMARY")
print("="*80)
print(summary.to_string(index=False))

# Best policy per task
best_per_task = (
    results_df.groupby(["Task", "Policy"])["Score"]
    .mean()
    .reset_index()
    .sort_values("Score", ascending=False)
    .groupby("Task")
    .first()
)
print("\n=== BEST POLICY PER TASK ===")
print(best_per_task[["Policy", "Score"]].round(3).to_string())

In [ ]:
# ── Save final results ─────────────────────────────────────────────────────────
results_df.to_csv("benchmark_results.csv", index=False)
summary.to_csv("benchmark_summary.csv",    index=False)
print("[OK] Results saved: benchmark_results.csv, benchmark_summary.csv")

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                     TRAINING PIPELINE COMPLETE                          ║
║                                                                          ║
║  Artifacts produced:                                                     ║
║  • bc_policy.pt              — Behavioral cloning warm-start weights     ║
║  • PPO_beta_suppression.zip  — PPO on easy task                          ║
║  • SAC_beta_suppression.zip  — SAC on easy task                          ║
║  • TD3_beta_suppression.zip  — TD3 on easy task                          ║
║  • curriculum_{algo}_*.zip   — Curriculum-trained best model             ║
║  • benchmark_results.csv     — All episode results                       ║
║  • benchmark_summary.csv     — Aggregated per-task statistics            ║
║  • *.png                     — Training curves and analysis plots        ║
║                                                                          ║
║  Next steps:                                                             ║
║  1. Increase N_EXPERT_EPS and TRAIN_STEPS for higher-quality training    ║
║  2. Add LSTM backbone for temporal memory (beta_trend, stim_washout)     ║
║  3. Fine-tune on expert tasks (fragile, refractory, generalization)      ║
║  4. Add Lagrangian SAC constraint on side_effect_load for fragile task   ║
╚══════════════════════════════════════════════════════════════════════════╝
""")